# Results for model: mistral-large-3:675b

In [ ]:
import polars as pl
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

# Load data
df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

# Feature Engineering: Calculate global TOP_QUANTILE (top 15%) of 'feature_00' relative to 'responder_6'
def calculate_top_quantile(df, feature_col, responder_col, quantile=0.85):
    quantile_threshold = df.select(
        pl.col(feature_col).quantile(quantile).over("date_id")
    ).unique().to_series().quantile(0.85)
    return df.with_columns(
        pl.when(pl.col(feature_col) >= quantile_threshold)
        .then(1)
        .otherwise(0)
        .alias("top_quantile_feature")
    )

df = calculate_top_quantile(df, "feature_00", "responder_6")

# Prepare features and target
features = [col for col in df.columns if col.startswith("feature_")]
features.append("top_quantile_feature")
target = "responder_6"

X = df.select(features).to_numpy()
y = df.select(target).to_numpy().ravel()

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost Regressor
model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="hist"
)

model.fit(X_train, y_train)

# Evaluate
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print(f"Train R²: {train_score:.4f}, Test R²: {test_score:.4f}")